In [0]:

# SILVER LAYER (Part 2) — Daily Market Clean

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, DoubleType, IntegerType

In [0]:
# ── 0. CONFIG ────────────────────────────────────────────────
spark.sql("USE CATALOG iran_israel_capstone_project")
spark.sql("USE SCHEMA bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

START_DATE = "2023-10-01"
END_DATE   = "2025-03-31"

In [0]:
# STEP 1 — BUILD TRADING CALENDAR (NSE days only)

# Generate every calendar day then keep only Mon-Fri.
# We will drop remaining NSE holidays by inner-joining with
# actual market_data rows that have real close prices.

# NSE public holidays in the date range (exchange-published list)
nse_holidays = [
    "2023-10-02", "2023-10-24", "2023-11-14", "2023-11-27",
    "2023-12-25",
    "2024-01-22", "2024-01-26", "2024-03-25", "2024-03-29",
    "2024-04-14", "2024-04-17", "2024-04-21", "2024-05-23",
    "2024-06-17", "2024-07-17", "2024-08-15", "2024-10-02",
    "2024-10-14", "2024-11-01", "2024-11-15", "2024-11-20",
    "2024-12-25",
    "2025-02-26", "2025-03-14", "2025-03-31",
]

In [0]:
# Build calendar from market_data itself — if ^NSEI traded, it's a valid day
trading_calendar = (
    spark.table("bronze.market_data")
    .filter(F.col("ticker") == "^NSEI")
    .filter(F.col("close").isNotNull())
    .select(F.to_date("trade_date").alias("trade_date"))
    .distinct()
    .filter(F.col("trade_date").between(START_DATE, END_DATE))
    .filter(~F.col("trade_date").isin(nse_holidays))          # drop holidays
    .filter(F.dayofweek("trade_date").isin([2,3,4,5,6]))      # Mon-Fri only
    .orderBy("trade_date")
)

In [0]:
print(f"Trading days in calendar: {trading_calendar.count()}")

TICKER_COLS = {
    "^NSEI"           : "nifty",
    "^BSESN"          : "sensex",
    "^NSEBANK"        : "niftybank",
    "NIFTYENERGY.NS"  : "niftyenergy",
    "^CNXAUTO"        : "niftyauto",
    "^CNXIT"          : "niftyit",
    "HAL.NS"          : "hal",
    "INDIGO.NS"       : "indigo",
    "ASIANPAINT.NS"   : "asianpaint",
    "ONGC.NS"         : "ongc",
    "BZ=F"            : "brent",
    "INR=X"           : "usdinr",
    "GC=F"            : "gold",
    "^INDIAVIX"       : "indiavix",
}

In [0]:
raw_mkt = (
    spark.table("bronze.market_data")
    .select(
        F.to_date("trade_date").alias("trade_date"),
        F.col("close").cast(DoubleType()),
        F.col("ticker"),
    )
    .filter(F.col("ticker").isin(list(TICKER_COLS.keys())))
    .filter(F.col("trade_date").between(START_DATE, END_DATE))
)

In [0]:
# Pivot: rows = trade_date, columns = one close per ticker
pivot_df = (
    raw_mkt
    .groupBy("trade_date")
    .pivot("ticker", list(TICKER_COLS.keys()))
    .agg(F.first("close"))
)

In [0]:
# Rename columns to friendly names
for orig, friendly in TICKER_COLS.items():
    col_alias = f"{friendly}_close"
    pivot_df = pivot_df.withColumnRenamed(orig, col_alias)

In [0]:
# Keep only valid NSE trading days
pivot_df = pivot_df.join(trading_calendar, on="trade_date", how="inner")

print(f"Pivoted rows (should ≈ trading days): {pivot_df.count()}")

In [0]:

# STEP 3 — DERIVE DAILY & ROLLING RETURNS

# Window ordered by trade_date for lag/rolling calculations
date_win       = Window.orderBy("trade_date")
rolling_5_win  = Window.orderBy("trade_date").rowsBetween(-4, 0)   # 5 trading days
rolling_20_win = Window.orderBy("trade_date").rowsBetween(-19, 0)  # 20 trading days

In [0]:
def daily_return(close_col: str) -> F.Column:
    """(close - prev_close) / prev_close * 100"""
    prev = F.lag(close_col, 1).over(date_win)
    return ((F.col(close_col) - prev) / prev * 100).cast(DoubleType())

def rolling_return(close_col: str, window) -> F.Column:
    """Compound return over a rolling window using first/last close."""
    first_close = F.first(close_col).over(window)
    last_close  = F.last(close_col).over(window)
    return ((last_close - first_close) / first_close * 100).cast(DoubleType())

In [0]:
silver_returns = (
    pivot_df
    # ── Nifty returns 
    .withColumn("nifty_daily_return_pct", daily_return("nifty_close"))
    .withColumn("nifty_5d_return_pct", rolling_return("nifty_close", rolling_5_win))
    .withColumn("nifty_20d_return_pct", rolling_return("nifty_close", rolling_20_win))
    # ── Brent daily change
    .withColumn("brent_daily_change_pct", daily_return("brent_close"))
    # ── USD/INR daily change (positive = rupee weakening) ────
    .withColumn("usdinr_daily_change_pct", daily_return("usdinr_close"))
    # ── India VIX 20-day moving average (for Gold G5) 
    .withColumn(
        "indiavix_20d_ma",
        F.avg("indiavix_close").over(rolling_20_win).cast(DoubleType()),
    )
    # ── Sector stock daily returns ───────────────────────────
    .withColumn("hal_return_pct", daily_return("hal_close"))
    .withColumn("indigo_return_pct", daily_return("indigo_close"))
    .withColumn("asianpaint_return_pct", daily_return("asianpaint_close"))
    .withColumn("ongc_return_pct", daily_return("ongc_close"))
    .withColumn("sensex_return_pct", daily_return("sensex_close"))
    .withColumn("niftybank_return_pct", daily_return("niftybank_close"))
    .withColumn("niftyenergy_return_pct", daily_return("niftyenergy_close"))
    .withColumn("niftyauto_return_pct", daily_return("niftyauto_close"))
    .withColumn("niftyit_return_pct", daily_return("niftyit_close"))
    .withColumn("gold_return_pct", daily_return("gold_close"))
)

In [0]:
# ============================================================
# READ IN EVENT DIMENSION (Created in Notebook A)
# ============================================================
event_dim = spark.table("silver.event_dim")

In [0]:

# STEP 5 — JOIN events onto daily market data

# Left join so non-event days get NULL event columns (correct behaviour)
# silver_with_events = (
#     silver_returns
#     .join(
#         event_dim.select("event_date","event_id","event_type",
#                          "severity","crude_risk","t_plus_1_expected"),
#         on=silver_returns["trade_date"] == event_dim["event_date"],
#         how="left"
#     )
#     .drop("event_date")
# )

silver_with_events = (
    silver_returns
    .join(
        F.broadcast(
            event_dim.select("event_date","event_id","event_type",
                             "severity","crude_risk","t_plus_1_expected")
        ),
        on=silver_returns["trade_date"] == event_dim["event_date"],
        how="left"
    )
    .drop("event_date")
)

In [0]:
# KPI CHECK: all event dates must resolve (non-null event_id on event days)
event_dates = [r.event_date for r in event_dim.select("event_date").collect()]
unmatched = (
    silver_with_events
    .filter(F.col("trade_date").isin(event_dates))
    .filter(F.col("event_id").isNull())
    .count()
)
print(f"Event join unmatched rows (must be 0): {unmatched}")
assert unmatched == 0, "x Some event dates did not join to Silver!"
print(" 100% event join coverage confirmed")

In [0]:
# 
# STEP 6 — DERIVE days_since_last_event
# 
# Cast event dates to unix timestamp for arithmetic
event_dates_df = (
    event_dim.select(
        F.col("event_date").alias("ev_date"),
        F.unix_timestamp("event_date").alias("ev_ts")
    )
)

In [0]:
# For each trade_date, find the most recent event_date <= trade_date
# We use a cross join + filter + window to get the latest prior event
silver_with_ts = silver_with_events.withColumn(
    "trade_ts", F.unix_timestamp("trade_date")
)

In [0]:
days_since = (
    silver_with_ts
    .crossJoin(event_dates_df)
    .filter(F.col("ev_ts") <= F.col("trade_ts"))
    .withColumn(
        "days_diff",
        ((F.col("trade_ts") - F.col("ev_ts")) / 86400).cast(IntegerType())
    )
    .groupBy("trade_date")
    .agg(F.min("days_diff").alias("days_since_last_event"))
)

In [0]:
silver_with_events = (
    silver_with_events
    .join(days_since, on="trade_date", how="left")
)

In [0]:
# 
# STEP 7 — JOIN FII / DII flows
# 
fii_clean = (
    spark.table("bronze.fii_raw")
    .select(
        F.col("date").cast(DateType()).alias("trade_date"),
        F.col("fii_net_buy_sell_cr").cast(DoubleType()).alias("fii_net_cr"),
        F.col("fii_gross_buy_cr").cast(DoubleType()).alias("fii_gross_buy_cr"),
        F.col("fii_gross_sell_cr").cast(DoubleType()).alias("fii_gross_sell_cr"),
    )
    .filter(F.col("trade_date").isNotNull())
)

In [0]:
# DII net is not in raw — we derive it where available; mark NULLs
# (DII data is absent from schema, so we flag and document)
fii_clean = fii_clean.withColumn(
    "dii_net_cr",
    F.lit(None).cast(DoubleType())   # documented: DII not in source feed
)

silver_with_fii = (
    silver_with_events
    .join(fii_clean, on="trade_date", how="left")
)

In [0]:
# 
# FII Forward Fill — reduce FII null rate to < 2%

# Some trading days have no FII data (weekends/holidays in FII feed
# that differ from NSE calendar). Forward-fill from last known value.
from pyspark.sql import DataFrame

def forward_fill(df: DataFrame, cols: list) -> DataFrame:
    for col in cols:
        win = Window.orderBy("trade_date").rowsBetween(Window.unboundedPreceding, 0)
        df = df.withColumn(
            f"{col}_ffill",
            F.last(col, ignorenulls=True).over(win)
        ).drop(col).withColumnRenamed(f"{col}_ffill", col)
    return df

fii_cols = ["fii_net_cr", "fii_gross_buy_cr", "fii_gross_sell_cr"]
silver_with_fii = forward_fill(silver_with_fii, fii_cols)
print("✅ Forward fill applied to FII columns")

In [0]:
# here forward fill is totally valid because:
# we are analyzing:

# trends
# correlations
# not exact tick-level execution

In [0]:
# NULL-rate check on FII
fii_null_rate = (
    silver_with_fii.filter(F.col("fii_net_cr").isNull()).count()
    / silver_with_fii.count() * 100
)
print(f"FII null rate: {fii_null_rate:.2f}%  (target < 2%)")

In [0]:

# STEP 8 — JOIN Brent from Alpha Vantage macro (cross-validate)

brent_macro = (
    spark.table("bronze.macro_brent_alpha_raw")
    .select(
        F.col("record_date").alias("trade_date"),
        F.expr("try_cast(value as double)").alias("brent_alpha_close"),
    )
    .filter(F.col("trade_date").isNotNull())
    .filter(F.col("brent_alpha_close").isNotNull())
)

In [0]:
silver_final = (
    silver_with_fii
    .join(brent_macro, on="trade_date", how="left")
    # Reconciliation column: use Yahoo brent when available, fall back to Alpha Vantage
    .withColumn(
        "brent_reconciled_close",
        F.coalesce(F.col("brent_close"), F.col("brent_alpha_close"))
    )
)

In [0]:
# Fill missing values in critical columns using forward fill (last valid value)
critical_cols = ["nifty_close", "brent_close", "usdinr_close", "indiavix_close"]
silver_final = forward_fill(silver_final, critical_cols)

In [0]:
# Null rate check after forward fill
total_rows = silver_final.count()
print("\n── Null Rate Check After Forward Fill (target < 2%) ──")
for col in critical_cols:
    null_count = silver_final.filter(F.col(col).isNull()).count()
    pct = null_count / total_rows * 100
    status = "" if pct < 2 else "x"
    print(f"  {status} {col}: {pct:.3f}% null  ({null_count}/{total_rows} rows)")

In [0]:

# STEP 9 — NULL RATE CHECKS (Silver KPIs)

# Rebuild silver_final: Cell 21 uses .cast(DoubleType()) which fails on
# malformed values ('.') in bronze.brent_crude_raw.value.
# Use try_cast to return NULL instead of throwing.
brent_macro = (
    spark.table("bronze.macro_brent_alpha_raw")
    .select(
        F.col("record_date").alias("trade_date"),
        F.expr("try_cast(value as double)").alias("brent_alpha_close"),
    )
    .filter(F.col("trade_date").isNotNull())
    .filter(F.col("brent_alpha_close").isNotNull())
)

In [0]:
silver_final = (
    silver_with_fii
    .join(brent_macro, on="trade_date", how="left")
    .withColumn(
        "brent_reconciled_close",
        F.coalesce(F.col("brent_close"), F.col("brent_alpha_close"))
    )
)

In [0]:
#  new edit for to eliminate brent_close and indiavix close null values so that they are less than 0.5 percent each
# Forward-fill brent_close and indiavix_close to bring nulls < 0.5%
silver_final = forward_fill(silver_final, ["brent_close", "indiavix_close"])

critical_cols = ["nifty_close", "brent_close", "usdinr_close", "indiavix_close"]
total_rows = silver_final.count()

In [0]:
print("\n── Null Rate Check (target < 0.5% each) ──")
for col in critical_cols:
    null_count = silver_final.filter(F.col(col).isNull()).count()
    pct = null_count / total_rows * 100
    status = "" if pct < 0.5 else ""
    print(f"  {status} {col}: {pct:.3f}% null  ({null_count}/{total_rows} rows)")

In [0]:

# STEP 10 — NON-TRADING DAY EXCLUSION CHECK

weekend_rows = (
    silver_final
    .filter(F.dayofweek("trade_date").isin([1, 7]))  # 1=Sun, 7=Sat
    .count()
)
print(f"\nWeekend rows in Silver (must be 0): {weekend_rows}")
assert weekend_rows == 0, "❌ Weekend rows found in Silver!"
print("✅ No weekend rows")

In [0]:

# STEP 11 — ADD SILVER METADATA & FINAL COLUMN ORDER

silver_final = (
    silver_final
    .withColumn("silver_timestamp", F.current_timestamp())
    .select(
        # ── Time dimension 
        "trade_date",

        # ── Nifty 
        "nifty_close",
        "nifty_daily_return_pct",
        "nifty_5d_return_pct",
        "nifty_20d_return_pct",

        # ── Other indices 
        "sensex_close", "sensex_return_pct",
        "niftybank_close", "niftybank_return_pct",
        "niftyenergy_close", "niftyenergy_return_pct",
        "niftyauto_close", "niftyauto_return_pct",
        "niftyit_close", "niftyit_return_pct",

        # ── Macro 
        "brent_close", "brent_alpha_close", "brent_reconciled_close",
        "brent_daily_change_pct",
        "usdinr_close", "usdinr_daily_change_pct",
        "gold_close", "gold_return_pct",
        "indiavix_close", "indiavix_20d_ma",

        # ── FII / DII
        "fii_net_cr", "fii_gross_buy_cr", "fii_gross_sell_cr",
        "dii_net_cr",

        # ── Sector stocks 
        "hal_close", "hal_return_pct",
        "indigo_close", "indigo_return_pct",
        "asianpaint_close", "asianpaint_return_pct",
        "ongc_close", "ongc_return_pct",

        # ── Event join 
        "event_id", "event_type", "severity",
        "crude_risk", "t_plus_1_expected",
        "days_since_last_event",

        # ── Audit 
        "silver_timestamp",
    )
)

In [0]:

# STEP 12 — WRITE silver.daily_market_clean

(
    silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("trade_date")
    .saveAsTable("silver.daily_market_clean")
)
print("\n✅ silver.daily_market_clean written successfully")

In [0]:

# STEP 13 — RETURN CALCULATION SPOT-CHECK (KPI: 10/10 match)

print("\n── Return Calculation Spot-Check ──")

spot_check_dates = [
    "2024-04-14", "2024-04-15", "2024-04-16",
    "2024-10-01", "2024-10-02", "2024-10-03",
    "2024-01-15", "2024-06-10", "2025-01-20", "2025-02-14",
]

In [0]:
df_check = spark.table("silver.daily_market_clean").filter(
    F.col("trade_date").isin(spot_check_dates)
).select("trade_date", "nifty_close", "nifty_daily_return_pct").orderBy("trade_date")

In [0]:
# Compare with manual formula using lag in a separate computation
manual_check = (
    spark.table("silver.daily_market_clean")
    .select("trade_date", "nifty_close")
    .orderBy("trade_date")
    .withColumn("prev_close", F.lag("nifty_close", 1).over(date_win))
    .withColumn(
        "manual_return",
        ((F.col("nifty_close") - F.col("prev_close")) / F.col("prev_close") * 100)
        .cast(DoubleType())
    )
    .filter(F.col("trade_date").isin(spot_check_dates))
)

In [0]:
joined_check = (
    df_check.alias("s")
    .join(manual_check.alias("m"), on="trade_date")
    .withColumn("diff", F.abs(F.col("s.nifty_daily_return_pct") - F.col("m.manual_return")))
    .withColumn("pass", F.col("diff") < 0.0001)
)

In [0]:
print("\nSpot-check results:")
joined_check.select(
    "trade_date",
    F.round("s.nifty_daily_return_pct", 6).alias("silver_return"),
    F.round("m.manual_return", 6).alias("manual_return"),
    F.round("diff", 8).alias("diff"),
    "pass"
).show(10, truncate=False)

In [0]:
pass_count = joined_check.filter(F.col("pass") == True).count()
print(f"Spot-check pass rate: {pass_count}/{joined_check.count()} (target: 10/10)")

In [0]:

# STEP 14 — days_since_last_event SPOT-CHECK (KPI: 5/5)

print("\n── days_since_last_event Spot-Check ──")

spot_5 = ["2024-04-15", "2024-04-22", "2024-10-03",
          "2024-10-28", "2025-01-10"]

spark.table("silver.daily_market_clean") \
    .filter(F.col("trade_date").isin(spot_5)) \
    .select("trade_date", "event_id", "days_since_last_event") \
    .orderBy("trade_date") \
    .show(truncate=False)

In [0]:

# STEP 15 — FINAL SILVER KPI SUMMARY

total = spark.table("silver.daily_market_clean").count()
events_joined = spark.table("silver.daily_market_clean") \
    .filter(F.col("event_id").isNotNull()).count()

print("\n══════════════════════════════════════════════")
print("         SILVER LAYER — KPI SUMMARY")
print("══════════════════════════════════════════════")
print(f"  Total trading days           : {total}")
print(f"  Event days with event_id     : {events_joined}")
print(f"  Weekend rows (must be 0)     : {weekend_rows}")
print(f"  Event join unmatched (0)     : {unmatched}")
print(f"  FII null rate                : {fii_null_rate:.2f}%")
print(f"  Return spot-check            : {pass_count}/10")
print("══════════════════════════════════════════════")